# BETO + BiLSTM, standalone Google Drive notebook
This notebook needs only `beto_rf_rnf.zip` and `Dataset6000Req.xlsx`. It does not require the repository on Drive. Execute cells with `Shift+Enter`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from google.colab import files
from zipfile import BadZipFile
import hashlib, json, random, re, shutil, subprocess, sys, time, unicodedata, zipfile
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/Protoype-Elbeto')
ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'openpyxl', 'deep-translator', 'transformers>=4.40,<6', 'scikit-learn', 'accelerate'], check=True)

def valid_zip(path):
    try:
        with zipfile.ZipFile(path) as archive: return archive.testzip() is None
    except (OSError, BadZipFile): return False

# Copy from mounted Drive to local disk before opening. Drive can return transient transport errors.
zip_candidates = [ROOT / 'beto_rf_rnf.zip', ROOT / 'models/beto_rf_rnf.zip']
drive_zip = next((path for path in zip_candidates if path.exists()), None)
local_artifact_zip = Path('/content/beto_rf_rnf.zip')
if drive_zip is not None:
    try: shutil.copy2(drive_zip, local_artifact_zip)
    except OSError as error: print('Drive ZIP copy failed:', error)
if not valid_zip(local_artifact_zip):
    print('Drive ZIP unavailable or invalid. Upload a fresh beto_rf_rnf.zip.')
    uploaded = files.upload(); names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not names: raise FileNotFoundError('A valid beto_rf_rnf.zip file is required.')
    local_artifact_zip = Path('/content') / names[0]
    if not valid_zip(local_artifact_zip): raise BadZipFile('Uploaded file is not a valid ZIP archive.')
    try: shutil.copy2(local_artifact_zip, ROOT / 'beto_rf_rnf.zip')
    except OSError: print('Could not save ZIP to Drive; continuing locally.')

artifact_staging = Path('/content/beto_rf_rnf_extract')
shutil.rmtree(artifact_staging, ignore_errors=True); artifact_staging.mkdir()
with zipfile.ZipFile(local_artifact_zip) as archive: archive.extractall(artifact_staging)
model_files = list(artifact_staging.rglob('model.safetensors'))
if not model_files: raise FileNotFoundError('ZIP does not contain model.safetensors.')
INIT_MODEL = model_files[0].parent
print('Initial BETO:', INIT_MODEL)

xlsx_candidates = [ROOT / 'Dataset6000Req.xlsx', ROOT / 'training/Dataset6000Req.xlsx']
XLSX_PATH = next((path for path in xlsx_candidates if path.exists()), None)
if XLSX_PATH is None:
    print('Upload Dataset6000Req.xlsx now.')
    uploaded = files.upload(); names = [name for name in uploaded if name.lower().endswith(('.xlsx', '.xlsm'))]
    if not names: raise FileNotFoundError('Dataset6000Req.xlsx is required.')
    XLSX_PATH = ROOT / names[0]
    try: shutil.copy2('/content/' + names[0], XLSX_PATH)
    except OSError: XLSX_PATH = Path('/content') / names[0]

DATA_DIR = ROOT / 'beto_lstm_data'
OUTPUT_DIR = ROOT / 'beto_lstm_rf_rnf'
CHECKPOINT_DIR = ROOT / 'beto_lstm_checkpoints'
DATA_DIR.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('XLSX:', XLSX_PATH)
print('Device:', DEVICE)

## Translate and split
The source file is English. This cell translates every unique requirement to Spanish, caches translations on Drive, maps `FR -> 0` and `NFR -> 1`, removes duplicates, and creates stratified partitions.

In [ ]:
from deep_translator import GoogleTranslator
from sklearn.model_selection import train_test_split

def clean_text(value):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(value))).strip()

frame = pd.read_excel(XLSX_PATH)
text_column = next(column for column in frame.columns if 'requirement' in str(column).lower() or 'text' in str(column).lower())
label_column = next(column for column in frame.columns if str(column).strip().lower() in {'type', 'label', 'class', 'category'})
frame = pd.DataFrame({'text': frame[text_column].map(clean_text), 'label': frame[label_column].map(lambda value: 1 if str(value).strip().lower() == 'nfr' else 0)})
frame = frame[frame.text.str.len() >= 3].drop_duplicates('text').reset_index(drop=True)
if set(frame.label.unique()) != {0, 1}: raise ValueError('Expected FR and NFR labels.')

cache_path = ROOT / 'translation_cache.json'
cache = json.loads(cache_path.read_text(encoding='utf-8')) if cache_path.exists() else {}
translator = GoogleTranslator(source='en', target='es')
translated, fallback_rows = [], 0
for index, text in enumerate(frame.text.tolist(), start=1):
    key = hashlib.sha256(text.encode('utf-8')).hexdigest()
    if key not in cache:
        for attempt in range(3):
            try:
                cache[key] = clean_text(translator.translate(text))
                break
            except Exception:
                if attempt == 2:
                    cache[key] = text
                    fallback_rows += 1
                else:
                    time.sleep(2 ** attempt)
        cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding='utf-8')
    translated.append(cache[key])
    if translated[-1] == text: fallback_rows += 1 if key not in cache else 0
    if index % 100 == 0: print(f'Translated {index}/{len(frame)}')
frame.text = translated

train, remainder = train_test_split(frame, test_size=0.30, random_state=42, stratify=frame.label)
val, test = train_test_split(remainder, test_size=0.50, random_state=42, stratify=remainder.label)
for name, split in [('train', train), ('val', val), ('test', test)]: split.to_csv(DATA_DIR / f'{name}.csv', index=False)
report = {'rows': len(frame), 'fallback_rows': fallback_rows, 'labels': frame.label.value_counts().to_dict(), 'translated': True}
(DATA_DIR / 'report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print(pd.read_csv(DATA_DIR / 'train.csv').head())

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertModel
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

class BETOBiLSTM(nn.Module):
    def __init__(self, encoder, hidden_size=256, dropout=0.3):
        super().__init__(); self.encoder = encoder
        self.lstm = nn.LSTM(encoder.config.hidden_size, hidden_size, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout); self.classifier = nn.Linear(hidden_size * 2, 2)
    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        args = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if token_type_ids is not None: args['token_type_ids'] = token_type_ids
        sequence = self.encoder(**args).last_hidden_state
        lengths = attention_mask.sum(1).to(torch.int64).cpu().clamp_min(1)
        packed = nn.utils.rnn.pack_padded_sequence(sequence, lengths, batch_first=True, enforce_sorted=False)
        _, (_, hidden) = self.lstm(packed)
        logits = self.classifier(self.dropout(torch.cat((hidden[-2], hidden[-1]), dim=1)))
        loss = nn.functional.cross_entropy(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

class RequirementDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=128):
        self.labels = torch.tensor(frame.label.astype(int).tolist(), dtype=torch.long)
        self.encoded = tokenizer(frame.text.astype(str).tolist(), truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    def __len__(self): return len(self.labels)
    def __getitem__(self, index): return {**{key: value[index] for key, value in self.encoded.items()}, 'labels': self.labels[index]}

def evaluate(model, loader):
    model.eval(); labels=[]; predictions=[]; losses=[]
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}; output = model(**batch)
            losses.append(float(output.loss)); labels += batch['labels'].cpu().tolist(); predictions += output.logits.argmax(-1).cpu().tolist()
    return {'loss': float(np.mean(losses)), 'accuracy': accuracy_score(labels, predictions), 'precision_rnf': precision_score(labels, predictions, zero_division=0), 'recall_rnf': recall_score(labels, predictions, zero_division=0), 'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0), 'confusion_matrix': confusion_matrix(labels, predictions, labels=[0, 1]).tolist()}

set_seed(); tokenizer = AutoTokenizer.from_pretrained(INIT_MODEL, local_files_only=True)
frames = {name: pd.read_csv(DATA_DIR / f'{name}.csv') for name in ('train', 'val', 'test')}
datasets = {name: RequirementDataset(frame, tokenizer) for name, frame in frames.items()}
loaders = {'train': DataLoader(datasets['train'], batch_size=16, shuffle=True), 'val': DataLoader(datasets['val'], batch_size=16), 'test': DataLoader(datasets['test'], batch_size=16)}
encoder = BertModel.from_pretrained(INIT_MODEL, local_files_only=True)
model = BETOBiLSTM(encoder).to(DEVICE)
for parameter in model.encoder.embeddings.parameters(): parameter.requires_grad = False
for layer in model.encoder.encoder.layer[:9]:
    for parameter in layer.parameters(): parameter.requires_grad = False
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=2e-5, weight_decay=0.01)
print('Trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
best_state = None; best_f1 = -1; stale = 0; history = []; CHECKPOINT_DIR.mkdir(exist_ok=True)
for epoch in range(1, 7):
    model.train(); losses = []
    for batch in loaders['train']:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}; optimizer.zero_grad(set_to_none=True)
        output = model(**batch); output.loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step(); losses.append(float(output.loss))
    validation = evaluate(model, loaders['val']); record = {'epoch': epoch, 'train_loss': float(np.mean(losses)), 'validation': validation}; history.append(record); print(json.dumps(record, indent=2))
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch}, CHECKPOINT_DIR / f'epoch_{epoch}.pt')
    if validation['f1_macro'] > best_f1: best_f1 = validation['f1_macro']; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}; stale = 0
    else:
        stale += 1
        if stale >= 2: print('Early stopping'); break
model.load_state_dict(best_state); test_metrics = evaluate(model, loaders['test']); print('TEST:', json.dumps(test_metrics, indent=2))

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.encoder.save_pretrained(OUTPUT_DIR / 'base_encoder')
tokenizer.save_pretrained(OUTPUT_DIR)
torch.save({k: v.detach().cpu() for k, v in model.state_dict().items() if k.startswith(('lstm.', 'classifier.'))}, OUTPUT_DIR / 'head.pt')
metadata = {'architecture': 'BETO+BiLSTM', 'labels': {'0': 'RF', '1': 'RNF'}, 'max_length': 128, 'lstm_hidden_size': 256, 'dropout': 0.3, 'test': test_metrics, 'history': history}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(ROOT / 'beto_lstm_rf_rnf'), 'zip', root_dir=ROOT, base_dir='beto_lstm_rf_rnf')
print('Artifact:', OUTPUT_DIR)
print('ZIP:', archive)